# Analysis code for normal mode analysis and ClustENMD

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
from prody import *
import seaborn as sns
import pandas as pd
import MDAnalysis as mda
plt.ion()

## ANM analysis 

In [2]:
import numpy as np 
from prody import *
from pylab import *
from matplotlib.pyplot import hist
import pandas as pd

In [4]:
# starting structure for ANM analysis
gumk=parsePDB('../../ClustENM/starting_structure_fixed.pdb')

@> 5637 atoms and 1 coordinate set(s) were parsed in 0.06s.


In [5]:
gumk_anm = ANM('gumk ca')
gumk_anm

<ANM: gumk ca (0 modes; 0 nodes)>

In [6]:
gumk_ca = gumk.ca
gumk_ca

<Selection: 'ca' from starting_structure_fixed (360 atoms)>

In [12]:
gumk_anm.buildHessian(gumk_ca)
gumk_anm.calcModes(100)

@> Hessian was built in 0.10s.
@> 100 modes were calculated in 0.14s.


In [ ]:
saveModel(gumk_anm)

## Compute PCA for ClustENMD ensemble 

In [68]:
saved_clustenm = loadEnsemble('starting_structure_clustenm.ens.npz')

KeyboardInterrupt: 

In [73]:
saved_clustenm.select('ca')

In [74]:
pca_clustenm = PCA('gumk ca')
pca_clustenm.buildCovariance(saved_clustenm)
pca_clustenm.calcModes()

@> Covariance is calculated using 12745 coordinate sets.
@> Covariance matrix calculated in 0.352040s.
@> 20 modes were calculated in 0.24s.


In [75]:
saveModel(pca_clustenm)

'gumk_ca.pca.npz'

## Projection of ClustENMD ensamble on NMs space 

In [5]:
#u clustENMD ensemble
structure = parsePDB('./vacum_fes/ref_dcd.pdb')
dcd = parseDCD('./vacum_fes/ensamble.dcd')

@> 5637 atoms and 1 coordinate set(s) were parsed in 0.07s.
@> DCD file contains 1275 coordinate sets for 5637 atoms.
@> DCD file was parsed in 1.08 seconds.
@> 82.35 MB parsed at input rate 76.30 MB/s.
@> 1275 coordinate sets parsed at input rate 1181 frame/s.


In [6]:
dcd.setCoords(structure)

In [7]:
dcd.setAtoms(structure.calpha)

In [8]:
# reference structure to set the zero of the plot
ref_struct = parsePDB(f'{root}/GlucTrans/GumK/full_protein/WT/free_protein/Membrane/UMD/charmm-gui-2655871296/step1_pdbreader.pdb')

@> 5637 atoms and 1 coordinate set(s) were parsed in 0.05s.


In [9]:
def_vec = np.array([frame.getCoords() - ref_struct.ca.getCoords() for frame in dcd])  

In [10]:
def_vec = def_vec.reshape((def_vec.shape[0],def_vec.shape[1] * 3))

In [11]:
prj_anm= (1 / (dcd.numAtoms()*0.5) ) * np.dot(def_vec,gumk_anm[:3].getArray() )

In [14]:
prj_pca= (1 / (dcd.numAtoms()*0.5) ) * np.dot(def_vec,gumk_pca[:3].getArray() )

NameError: name 'gumk_pca' is not defined

In [12]:
ensemble_rmsd=np.array([calcRMSD(ref_struct.ca.getCoords(), frame.getCoords()) for frame in dcd])

In [15]:
df_nm = pd.DataFrame({'NM1': prj_anm[:,0], 'NM2': prj_anm[:,1],'NM3': prj_anm[:,2], 'RMSD': ensemble_rmsd})

In [17]:
df_nm.to_csv('clustenmd_nomarl_modes.csv', index=False, header=True)